Micromagnetic Standard Problem 5 involves simulating the dynamics of a magnetic vortex in a thin rectangular film under
the influence of a spin-polarized current. The problem is designed to test the accuracy of micromagnetic simulations in
capturing the complex behavior of vortex structures, including their motion and deformation. The rectangular film in
this example has dimensions of 100 nm × 100 nm × 10 nm, and the simulation tracks the position of the vortex core
as it evolves over time due to the combined effects of exchange interactions, demagnetization, and spin-transfer torques.

In [1]:
using MicroMagnetic
using Printf
using CairoMakie

@using_gpu()

Define a function to initialize a vortex roughly.

In [2]:
function init_fun(i, j, k, dx, dy, dz)
    x = i - 10
    y = j - 10
    r = sqrt(x^2 + y^2)
    if r < 2
        return (0, 0, 1)
    end
    return (y / r, -x / r, 0)
end

init_fun (generic function with 1 method)

Define a callback function to record vortex center coordinates.

In [3]:
function call_back_fun(sim, t)
    Rx, Ry = compute_guiding_center(sim; z=1)
    open("std5_center.txt", "a") do f
        write(f, @sprintf("%g  %g  %g\n", t, Rx, Ry))
    end
end

call_back_fun (generic function with 1 method)

Define simulation parameters.

In [4]:
args = (
    name = "std5",
    task_s = ["relax", "dynamics"],          # List of tasks to perform
    driver_s = ["SD", "LLG_STT"],            # List of drivers to use
    mesh = FDMesh(; nx=20, ny=20, nz=2, dx=5e-9, dy=5e-9, dz=5e-9), # Mesh configuration
    Ms = 8e5,                               # Saturation magnetization
    A = 1.3e-11,                            # Exchange constant
    demag = true,                           # Enable demagnetization
    m0 = init_fun,                          # Initial magnetization function
    alpha = 0.1,                            # Gilbert damping parameter
    ux = -72.438,                           # Effective current density
    beta = 0.05,                            # Nonadiabatic STT parameter
    steps = 160,                            # Number of steps for dynamics
    dt = 0.05ns,                            # Time step size
    stopping_dmdt = 0.01,                   # Stopping criterion for relaxation
    call_back = call_back_fun,              # Callback function for vortex center tracking
    dynamic_m_interval = 1                  # Save magnetization at every step
);

Run the simulation using `sim_with` function.

In [5]:
sim_with(args);

[ Info: MicroSim has been created.
[ Info: Exchange has been added.
[ Info: Running Driver : MicroMagnetic.EnergyMinimization{Float64}.
[ Info: max_dmdt is less than stopping_dmdt=0.01 @steps=173, Done!
[ Info: The driver LLG_STT is used!
[ Info: step =    0  t = 0.000000e+00
[ Info: step =    1  t = 5.000000e-11
[ Info: step =    2  t = 1.000000e-10
[ Info: step =    3  t = 1.500000e-10
[ Info: step =    4  t = 2.000000e-10
[ Info: step =    5  t = 2.500000e-10
[ Info: step =    6  t = 3.000000e-10
[ Info: step =    7  t = 3.500000e-10
[ Info: step =    8  t = 4.000000e-10
[ Info: step =    9  t = 4.500000e-10
[ Info: step =   10  t = 5.000000e-10
[ Info: step =   11  t = 5.500000e-10
[ Info: step =   12  t = 6.000000e-10
[ Info: step =   13  t = 6.500000e-10
[ Info: step =   14  t = 7.000000e-10
[ Info: step =   15  t = 7.500000e-10
[ Info: step =   16  t = 8.000000e-10
[ Info: step =   17  t = 8.500000e-10
[ Info: step =   18  t = 9.000000e-10
[ Info: step =   19  t = 9.500000e-10
[

Generate a movie for the vortex dynamics.

In [6]:
jld2movie("std5.jld2"; output="assets/std5.mp4", component='z', figsize=(400,400))

"assets/std5.mp4"

![](./assets/std5.mp4)

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*